In [1]:
import pandas as pd
import os

BASE = '/kaggle/input/datasets/nitishabharathi/cert-insider-threat/'  # we'll confirm this below

# Step 1: Find exact path
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/nitishabharathi/cert-insider-threat/psychometric.csv
/kaggle/input/datasets/nitishabharathi/cert-insider-threat/email.csv


In [ ]:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    # Step 2: Preview both files
email = pd.read_csv(BASE + 'email.csv', nrows=5)
psycho = pd.read_csv(BASE + 'psychometric.csv', nrows=5)

print("EMAIL COLUMNS:", email.columns.tolist())
print(email.head())
print("\nPSYCHO COLUMNS:", psycho.columns.tolist())
print(psycho.head())

EMAIL COLUMNS: ['id', 'date', 'user', 'pc', 'to', 'cc', 'bcc', 'from', 'size', 'attachments', 'content']
                         id                 date     user       pc  \
0  {R3I7-S4TX96FG-8219JWFF}  01/02/2010 07:11:45  LAP0338  PC-5758   
1  {R0R9-E4GL59IK-2907OSWJ}  01/02/2010 07:12:16  MOH0273  PC-6699   
2  {G2B2-A8XY58CP-2847ZJZL}  01/02/2010 07:13:00  LAP0338  PC-5758   
3  {A3A9-F4TH89AA-8318GFGK}  01/02/2010 07:13:17  LAP0338  PC-5758   
4  {E8B7-C8FZ88UF-2946RUQQ}  01/02/2010 07:13:28  MOH0273  PC-6699   

                                                  to  \
0  Dean.Flynn.Hines@dtaa.com;Wade_Harrison@lockhe...   
1                        Odonnell-Gage@bellsouth.net   
2                         Penelope_Colon@netzero.com   
3                          Judith_Hayden@comcast.net   
4  Bond-Raymond@verizon.net;Alea_Ferrell@msn.com;...   

                                cc                          bcc  \
0  Nathaniel.Hunter.Heath@dtaa.com                          NaN   
1  

In [3]:
# ── Cell 2: Load files ─────────────────────────────────────────────────────
BASE = '/kaggle/input/datasets/nitishabharathi/cert-insider-threat/'   # update if path differs above

email  = pd.read_csv(BASE + 'email.csv', parse_dates=['date'],
                     dayfirst=False, infer_datetime_format=True)
psycho = pd.read_csv(BASE + 'psychometric.csv')

print(f"Email shape : {email.shape}")
print(f"Psycho shape: {psycho.shape}")

/tmp/ipykernel_17/1198156409.py:4: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  email  = pd.read_csv(BASE + 'email.csv', parse_dates=['date'],


Email shape : (2629979, 11)
Psycho shape: (1000, 7)


In [4]:
# ── Cell 3: Feature engineering ────────────────────────────────────────────

# ── Date parts ──
email['day']  = email['date'].dt.date
email['hour'] = email['date'].dt.hour

# ── Detect external recipients ──
# Internal domain = dtaa.com (seen in the data)
email['to_external'] = email['to'].str.lower().str.contains(
    r'(?!.*@dtaa\.com)[a-z0-9._%+\-]+@[a-z0-9.\-]+\.[a-z]{2,}',
    regex=True, na=False).astype(int)

email['from_personal'] = email['from'].str.lower().str.contains(
    'gmail|yahoo|hotmail|earthlink|comcast|verizon|msn|'
    'bellsouth|optonline|netzero|aol',
    regex=True, na=False).astype(int)

email['has_bcc']       = email['bcc'].notna().astype(int)
email['after_hours']   = ((email['hour'] < 7) | (email['hour'] > 19)).astype(int)

# ── Aggregate per user per day ──
daily = email.groupby(['user', 'day']).agg(
    email_count       = ('id',             'count'),
    to_external_count = ('to_external',    'sum'),
    from_personal_cnt = ('from_personal',  'sum'),
    bcc_count         = ('has_bcc',        'sum'),
    attachment_total  = ('attachments',    'sum'),
    avg_size          = ('size',           'mean'),
    after_hours_count = ('after_hours',    'sum'),
    unique_recipients = ('to',             'nunique'),
).reset_index()

# ── Ratio features (more meaningful than raw counts) ──
daily['external_ratio']  = daily['to_external_count'] / (daily['email_count'] + 1e-6)
daily['bcc_ratio']        = daily['bcc_count']        / (daily['email_count'] + 1e-6)
daily['after_hours_ratio']= daily['after_hours_count']/ (daily['email_count'] + 1e-6)
daily['personal_ratio']   = daily['from_personal_cnt']/ (daily['email_count'] + 1e-6)

print(f"Daily feature shape: {daily.shape}")
print(daily.head())

Daily feature shape: (326985, 14)
      user         day  email_count  to_external_count  from_personal_cnt  \
0  AAE0190  2010-01-04           14                  2                  1   
1  AAE0190  2010-01-05           13                  7                  5   
2  AAE0190  2010-01-06           14                  6                  6   
3  AAE0190  2010-01-07           14                  5                  5   
4  AAE0190  2010-01-08           13                  2                  2   

   bcc_count  attachment_total      avg_size  after_hours_count  \
0          0                 4  31523.428571                  0   
1          0                 2  27350.153846                  0   
2          0                12  38046.214286                  0   
3          0                 9  33902.214286                  0   
4          0                10  30818.230769                  0   

   unique_recipients  external_ratio  bcc_ratio  after_hours_ratio  \
0                 14        0.

In [5]:
# ── Cell 4: Create threat labels ───────────────────────────────────────────
# Threat heuristic (no ground truth file available):
# Flag a user-day as threat if they show 3+ suspicious signals together

daily['threat'] = (
    (daily['external_ratio']   > 0.7) &   # 70%+ emails going outside
    (daily['bcc_ratio']        > 0.3) &   # hiding recipients
    (daily['attachment_total'] > 2  )     # sending files
).astype(int)

# Also flag personal account usage on high-attachment days
daily['threat'] = daily['threat'] | (
    (daily['from_personal_ratio'] if 'from_personal_ratio' in daily.columns
     else daily['personal_ratio'] > 0.5) &
    (daily['attachment_total'] > 3)
).astype(int)

print(f"\nThreat distribution:")
print(daily['threat'].value_counts())
print(f"Threat rate: {daily['threat'].mean():.2%}")


Threat distribution:
threat
0    292423
1     34562
Name: count, dtype: int64
Threat rate: 10.57%


In [6]:
# ── Cell 5: Merge psychometric scores ──────────────────────────────────────
# Rename user_id to user to match email data
psycho = psycho.rename(columns={'user_id': 'user'})

# Add composite risk score: low C + low A = higher insider risk
psycho['risk_score'] = (100 - psycho['C']) + (100 - psycho['A'])

daily = daily.merge(psycho[['user','O','C','E','A','N','risk_score']],
                    on='user', how='left')

# Fill users not in psychometric with median
for col in ['O','C','E','A','N','risk_score']:
    daily[col] = daily[col].fillna(daily[col].median())

print(f"After merge: {daily.shape}")
print(daily.head())

After merge: (326985, 21)
      user         day  email_count  to_external_count  from_personal_cnt  \
0  AAE0190  2010-01-04           14                  2                  1   
1  AAE0190  2010-01-05           13                  7                  5   
2  AAE0190  2010-01-06           14                  6                  6   
3  AAE0190  2010-01-07           14                  5                  5   
4  AAE0190  2010-01-08           13                  2                  2   

   bcc_count  attachment_total      avg_size  after_hours_count  \
0          0                 4  31523.428571                  0   
1          0                 2  27350.153846                  0   
2          0                12  38046.214286                  0   
3          0                 9  33902.214286                  0   
4          0                10  30818.230769                  0   

   unique_recipients  ...  bcc_ratio  after_hours_ratio  personal_ratio  \
0                 14  ...        

In [7]:
# ── Cell 6: Sequence windowing for BiLSTM ──────────────────────────────────
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import numpy as np

WINDOW = 7   # 7-day window (smaller since fewer users/days than full CERT)

feature_cols = [
    'email_count', 'to_external_count', 'bcc_count',
    'attachment_total', 'avg_size', 'after_hours_count',
    'unique_recipients', 'external_ratio', 'bcc_ratio',
    'after_hours_ratio', 'personal_ratio',
    'O', 'C', 'E', 'A', 'N', 'risk_score'
]

# Normalize
scaler = MinMaxScaler()
daily[feature_cols] = scaler.fit_transform(daily[feature_cols].fillna(0))

X_seqs, y_seqs = [], []

for user, grp in daily.groupby('user'):
    grp = grp.sort_values('day').reset_index(drop=True)
    vals   = grp[feature_cols].values
    labels = grp['threat'].values

    if len(vals) < WINDOW:
        continue

    for i in range(len(vals) - WINDOW):
        X_seqs.append(vals[i : i + WINDOW])
        y_seqs.append(int(labels[i : i + WINDOW].any()))

X = np.array(X_seqs)
y = np.array(y_seqs)

print(f"X shape : {X.shape}  →  (samples, {WINDOW} days, {len(feature_cols)} features)")
print(f"y — normal: {(y==0).sum()}  threat: {(y==1).sum()}")

X shape : (319985, 7, 17)  →  (samples, 7 days, 17 features)
y — normal: 197702  threat: 122283


In [8]:
# ── Cell 7: Handle imbalance + train/test split ────────────────────────────
n_samples, n_steps, n_feats = X.shape
X_flat = X.reshape(n_samples, n_steps * n_feats)

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_flat, y)
X_res = X_res.reshape(-1, n_steps, n_feats)

X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)

print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")
print(f"Threat ratio in train: {y_train.mean():.2%}")

# Save for next phase
np.save('X_train.npy', X_train)
np.save('X_test.npy',  X_test)
np.save('y_train.npy', y_train)
np.save('y_test.npy',  y_test)

import joblib
joblib.dump(scaler, 'scaler.pkl')

print("\nAll arrays saved ✓  — ready for BiLSTM training")

Train : (316323, 7, 17)
Test  : (79081, 7, 17)
Threat ratio in train: 50.00%

All arrays saved ✓  — ready for BiLSTM training


In [9]:
# ── Cell 8: Install & imports for BiLSTM ──────────────────────────────────
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Bidirectional, LSTM,
                                     Dense, Dropout, BatchNormalization)
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                                        ModelCheckpoint)
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

# Load saved arrays
X_train = np.load('X_train.npy')
X_test  = np.load('X_test.npy')
y_train = np.load('y_train.npy')
y_test  = np.load('y_test.npy')

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"TensorFlow version: {tf.__version__}")

2026-03-26 04:45:23.740887: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774500323.984702      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774500324.053301      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774500324.682266      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774500324.682307      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774500324.682310      17 computation_placer.cc:177] computation placer alr

X_train: (316323, 7, 17)
X_test : (79081, 7, 17)
TensorFlow version: 2.19.0


In [10]:
# ── Cell 9: Build BiLSTM model ─────────────────────────────────────────────
def build_bilstm(timesteps, n_features):
    inp = Input(shape=(timesteps, n_features), name='input')

    # First BiLSTM layer — returns sequences for second layer
    x = Bidirectional(LSTM(128, return_sequences=True,
                           dropout=0.2, recurrent_dropout=0.2),
                      name='bilstm_1')(inp)
    x = BatchNormalization()(x)

    # Second BiLSTM layer — returns final state only
    x = Bidirectional(LSTM(64, return_sequences=False,
                           dropout=0.2, recurrent_dropout=0.2),
                      name='bilstm_2')(x)
    x = BatchNormalization()(x)

    # Dense layers
    x = Dense(64, activation='relu', name='dense_1')(x)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu', name='feature_layer')(x)  # ← DQN uses this
    x = Dropout(0.2)(x)

    out = Dense(1, activation='sigmoid', name='output')(x)

    model = Model(inputs=inp, outputs=out)
    return model

n_timesteps = X_train.shape[1]   # 7
n_features  = X_train.shape[2]   # 17

model = build_bilstm(n_timesteps, n_features)
model.summary()

2026-03-26 04:45:50.789763: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 7, 17)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_1 (Bidirectional)        │ (None, 7, 256)         │       149,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 7, 256)         │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_2 (Bidirectional)        │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_layer (Dense)           │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 325,761 (1.24 MB)

 Trainable params: 324,993 (1.24 MB)

 Non-trainable params: 768 (3.00 KB)

In [11]:
# ── Cell 10: Compile & train ───────────────────────────────────────────────
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)

callbacks = [
    EarlyStopping(monitor='val_auc', patience=5,
                  restore_best_weights=True, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('bilstm_best.h5', monitor='val_auc',
                    save_best_only=True, mode='max', verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=512,   # large batch = faster on big dataset
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete ✓")

Epoch 1/30
618/618 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.8111 - auc: 0.8896 - loss: 0.4106 - precision: 0.8111 - recall: 0.8124
Epoch 1: val_auc improved from -inf to 0.97457, saving model to bilstm_best.h5


618/618 ━━━━━━━━━━━━━━━━━━━━ 106s 149ms/step - accuracy: 0.8112 - auc: 0.8897 - loss: 0.4105 - precision: 0.8112 - recall: 0.8124 - val_accuracy: 0.9090 - val_auc: 0.9746 - val_loss: 0.2128 - val_precision: 0.9129 - val_recall: 0.9044 - learning_rate: 0.0010
Epoch 2/30
618/618 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.9153 - auc: 0.9746 - loss: 0.2004 - precision: 0.9022 - recall: 0.9320
Epoch 2: val_auc improved from 0.97457 to 0.99550, saving model to bilstm_best.h5


618/618 ━━━━━━━━━━━━━━━━━━━━ 90s 145ms/step - accuracy: 0.9153 - auc: 0.9746 - loss: 0.2004 - precision: 0.9023 - recall: 0.9320 - val_accuracy: 0.9623 - val_auc: 0.9955 - val_loss: 0.0936 - val_precision: 0.9535 - val_recall: 0.9720 - learning_rate: 0.0010
Epoch 3/30
618/618 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.9442 - auc: 0.9886 - loss: 0.1344 - precision: 0.9356 - recall: 0.9543
Epoch 3: val_auc improved from 0.99550 to 0.99636, saving model to bilstm_best.h5


618/618 ━━━━━━━━━━━━━━━━━━━━ 90s 146ms/step - accuracy: 0.9442 - auc: 0.9886 - loss: 0.1344 - precision: 0.9356 - recall: 0.9543 - val_accuracy: 0.9633 - val_auc: 0.9964 - val_loss: 0.0982 - val_precision: 0.9821 - val_recall: 0.9437 - learning_rate: 0.0010
Epoch 4/30
618/618 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - accuracy: 0.9507 - auc: 0.9911 - loss: 0.1188 - precision: 0.9447 - recall: 0.9575
Epoch 4: val_auc did not improve from 0.99636
618/618 ━━━━━━━━━━━━━━━━━━━━ 90s 146ms/step - accuracy: 0.9507 - auc: 0.9911 - loss: 0.1188 - precision: 0.9447 - recall: 0.9575 - val_accuracy: 0.9156 - val_auc: 0.9927 - val_loss: 0.1828 - val_precision: 0.9938 - val_recall: 0.8364 - learning_rate: 0.0010
Epoch 5/30
618/618 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - accuracy: 0.9567 - auc: 0.9932 - loss: 0.1042 - precision: 0.9533 - recall: 0.9607
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 5: val_auc did not improve from 0.99636
618/618 ━━━━━━━━━━━━━━━━━━━━ 94s 152m